# Generate Label Studio XML from Additional Needs Taxonomy

This notebook creates an XML frontend layout from the Addiotional Needs taxonomy based on a template.

## Environment Setup & Global Imports

In [1]:
import pandas as pd
import ast
from xml.sax.saxutils import escape

In [2]:
# Paths
XML_TEMPLATE_PATH = "../annotation/label-studio-ui-template.xml"
TAXONOMY_PATH = "../data/output/taxonomy_v2_autogen.csv"
OUTPUT_PATH = "../annotation/label-studio-ui.xml"

## Load Base XML Template & Taxnomgy CSV

In [3]:
with open(XML_TEMPLATE_PATH, 'r') as f:
    template = f.read().splitlines() 

In [4]:
taxonomy = pd.read_csv('../data/output/taxonomy_v2_autogen.csv')
taxonomy.head()

,high_level_category,category_description,values_hint,cat_label,regex
0,Care,Care experienced,"['Care leaver', 'Care experienced (under 25)']",care_care_experienced,\bcare experienced\b|\bcare leaver\b|\baged ou...
1,Care,Care setting,"['Fostered', 'Social care']",care_care_setting,\bfostered\b|\bfoster care\b|\bfoster placemen...
2,Care,Has caring responsibility,"['Formal', 'Informal']",care_has_caring_responsibility,\bformal carer\b|\bregistered carer\b|\bcarer....
3,Care,Social care involvement,"['Adult Social Care', ""Children's Social Care""]",care_social_care_involvement,\bsocial care\b|\bcare package\b|\bcare plan\b...
4,Cautions,ASBO or injunction obtained,"['ASBO', 'Injunction']",cautions_asbo_or_injunction_obtained,\bASBO\b|\binjunction\b|\bcivil injunction\b|\...


## Map High-Level Category Color Codes

To make the annotation UI easier to navigate for human reviewers, we map unique hex colors to each `high_level_category` group.

In [5]:
# Colour per high_level_category
GROUP_COLOURS = {
    'Care':                  '#58D68D',  # Bright Green
    'Cautions':              '#CD6155',  # Red
    'Reasonable Adjustments':"#7787EF",  # Dark Blue
    'Communications':        '#F4D03F',  # Bright Yellow
    'Disability':            "#D09DF6",  # Lilac
    'Health':                "#48C9C0",  # Turquoise
    'Housing Conditions':    "#F086F0",  # Pink
    'Life Events':           '#EB984E',  # Soft Orange
    'Mobility':              "#7FB3D5",  # Steel Blue
    'Property Level':        "#B5B9C2",  # Grey
    'Safety & Risk':         '#EC7063',  # Soft Red
}

DEFAULT_COLOUR = '#95a5a6'

## Generate Dynamic Label Elements

We iterate over each high-level taxonomy category group and automatically generate target standard XML `<Label>` string elements.

In [6]:
category_labels_xml = []

# Generate labels selection coloured by high_level_category
for high_level, group in taxonomy.groupby('high_level_category'):
    category_labels_xml.append('')
    colour = GROUP_COLOURS.get(str(high_level), DEFAULT_COLOUR)
    for row in group.itertuples():
        hint = ", ".join(ast.literal_eval(str(row.values_hint)))
        category_labels_xml.append(
            f'          <Label value="{row.cat_label}" html="{escape(str(row.category_description))}" background="{colour}" hint="{escape(hint)}"/>'
        )

category_labels_xml[:4]

['',
 '          <Label value="care_care_experienced" html="Care experienced" background="#58D68D" hint="Care leaver, Care experienced (under 25)"/>',
 '          <Label value="care_care_setting" html="Care setting" background="#58D68D" hint="Fostered, Social care"/>',
 '          <Label value="care_has_caring_responsibility" html="Has caring responsibility" background="#58D68D" hint="Formal, Informal"/>']

## Add the relation block

Construct a `<Relation>` block that allows `need_labels` to link (`AFFECTS`) to `entity_labels`.

In [7]:
relation = f'''      <Relation 
        value="AFFECTS" 
        fromName="need_labels" 
        toName="entity_labels" 
        label="{",".join(taxonomy['cat_label'].to_list())}"
      />'''

## Fill in Template & Export XML

Find the preset layout hooks (`<AN_PLACEHOLDER>` and `<RELATION_PLACEHOLDER>`) in the template, swap them with our generated arrays, and write the output configuration file.

In [8]:
needs_index = template.index("      <AN_PLACEHOLDER>")
new_file = template[:needs_index] + category_labels_xml + template[needs_index+1:]

relation_index = new_file.index("      <RELATION_PLACEHOLDER>")
new_file[relation_index] = relation

with open('../annotation/label-studio-ui.xml', 'w') as f:
    f.write('\n'.join(new_file))